# 外部ツールの統合

LLM に検索語を作らせ、Python で Wikipedia を検索し、取得した資料から回答させましょう。

以下の一連の流れを追います。

- LLM がツールを呼び出すコマンドを生成
- コマンドに応じてツールの出力を得る
- ツールの出力に基づいて LLM が回答する

In [1]:
!pip install -q google-genai  # Gemini API
!pip install -q requests  # Wikipedia との通信

## API の準備

[Google AI Studio](https://aistudio.google.com/apikey) で Gemini API キーを取得してください。キーの入力欄では、入力した文字は表示されません。
API の利用枠・料金も [Google AI Studio](https://aistudio.google.com/) で確認できます。

In [2]:
import os
from getpass import getpass
from google import genai

# 環境変数を優先し、なければ非表示入力で受け取る
api_key = os.environ.get("GEMINI_API_KEY") or getpass("Gemini API キー: ")
client = genai.Client(api_key=api_key)
model_name = "gemini-flash-latest"

Gemini API キー: ··········


## Wikipedia 検索

日本語版 Wikipedia の上位2記事から、冒頭の文章と URL を取得します。返り値は、状態を表す `status` と記事リストの `articles` を持つ辞書です。

In [3]:
import requests

def search_wikipedia_ja(query: str) -> dict:
    """日本語版 Wikipedia 上位2記事の冒頭と URL の取得"""
    try:
        # ja.wikipedia.org にエンドポイントを変更
        response = requests.get(
            "https://ja.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "format": "json",
                "formatversion": 2,
                "generator": "search",
                "gsrsearch": query,
                "gsrlimit": 2,
                "prop": "extracts|info",
                "inprop": "url",
                "exintro": 1,
                "explaintext": 1,
                "exchars": 800,
                "exlimit": "max",
            },
            timeout=20,
            headers={
                "User-Agent": "NLP-Illustrated Example Code (https://github.com/ryokan0123/nlp-illustrated-book)"
            },
        )
        response.raise_for_status()
        data = response.json()
        if "error" in data:
            return {"status": "api_error", "articles": [], "error": str(data["error"])}
    except (requests.RequestException, ValueError) as error:
        return {"status": "network_error", "articles": [], "error": str(error)}

    pages = sorted(
        data.get("query", {}).get("pages", []), key=lambda page: page.get("index", 999)
    )
    articles = [
        {
            "title": page["title"],
            "text": page.get("extract", "").strip(),
            "url": page["fullurl"],
        }
        for page in pages
        if page.get("extract")
    ]
    return {"status": "ok" if articles else "no_results", "articles": articles}

# 試しに検索
search_result_ja = search_wikipedia_ja("東京スカイツリー")
print("状態:", search_result_ja["status"])
for article in search_result_ja["articles"]:
    print(article["title"], "\n", article["text"], "\n", article["url"])

状態: ok
東京スカイツリー 
 東京スカイツリー（とうきょうスカイツリー、英: TOKYO SKYTREE）は、東京都墨田区押上にある電波塔（送信所）。2012年2月29日に完成し、同年5月22日に電波塔・観光施設として開業した。
東京のランドマークの一つ。高さは634 mで、タワーとしては世界第1位。建築物としてはブルジュ・ハリファ、ムルデカ118（PNB118）に次ぐ世界第3位となる。
商業施設「東京ソラマチ」やオフィスビル「東京スカイツリーイーストタワー」が併設されている。東武鉄道の旧：業平橋駅（現：とうきょうスカイツリー駅）の貨物駅跡地に建設された東京スカイツリータウンを構成する中核施設である。とうきょうスカイツリー駅・押上駅と直結している。
東武鉄道子会社の東武タワースカイツリー株式会社によって建設・運営されており、東武グループのシンボル的存在である。東京スカイツリーについての名称・ロゴマーク・シルエットデザイン・完成予想コンピュータグラフィックスといった知的財産は東武タワースカイツリー株式会社等の著作権および商標権により保護され、同社の登録商標にもなっているため、公式案内では「東京スカイツリー®」のように「®マーク」も併記されている。 
 https://ja.wikipedia.org/wiki/%E6%9D%B1%E4%BA%AC%E3%82%B9%E3%82%AB%E3%82%A4%E3%83%84%E3%83%AA%E3%83%BC
東京スカイツリータウン 
 東京スカイツリータウン（とうきょうスカイツリータウン、Tokyo Skytree Town）は、東京都墨田区押上に所在する、東京スカイツリーを中心とする複合商業施設。2012年（平成24年）5月22日開業。東武鉄道が運営する。
とうきょうスカイツリー駅と押上駅に直結し、両駅を囲む東西長さ約400mのエリアに位置する。東京スカイツリーを中心に、商業施設「東京ソラマチ」とオフィスビル「東京スカイツリーイーストタワー」からなり、東京ソラマチには「すみだ水族館」と「コニカミノルタプラネタリウム天空 in 東京スカイツリータウン」なども入居する。 
 https://ja.wikipedia.org/wiki/%E6%9D%B1%E4%BA%AC%E3%82%B9%E3%82%AB%E3%82%

## ツールの指定

ツール名と引数の書式をモデルに渡します。

In [4]:
from google.genai import types

# 検索ツールの仕様を LLM に伝えるための定義
declaration_ja = {
    "name": "search_wikipedia_ja",
    "description": "日本語のWikipediaから記事の要約とソースURLを検索します。",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "日本語の短い検索クエリ。"}
        },
        "required": ["query"],
    },
}
tool_ja = types.Tool(function_declarations=[declaration_ja])
config = types.GenerateContentConfig(
    tools=[tool_ja],
    # ツールの実行をこちらで制御するためのオプション
    # disable=False の場合は、ライブラリが裏側で関数の実行を行います
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    system_instruction="知識に関する質問を受けた場合は、回答する前に、必ず search_wikipedia_ja を呼び出してください。",
    max_output_tokens=512,
)


## 呼び出し要求から回答まで

### 検索語の生成

今回の検索語をモデルに作らせます。質問と回答は日本語で扱います。

In [5]:
question = "ドラえもんの作者の誕生日は？"
contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
response = client.models.generate_content(
    model=model_name, contents=contents, config=config
)
call = response.function_calls[0] if response.function_calls else None
print("モデルのツール呼び出しコマンド:")
print(call)

モデルのツール呼び出しコマンド:
id='call_538312' args={'query': '藤子・F・不二雄'} name='search_wikipedia_ja' partial_args=None will_continue=None


`call` に `search_wikipedia_ja` の呼び出し要求が表示されたことを確認してから、次のセルへ進みます。`None` の場合は、この後の検索処理を実行できません。

要求の `name` がツール名、`args` が引数です。`args={"query": "藤子・F・不二雄"}` なら、その文字列で検索します。


### Python による検索

ツール名と引数を確認してから、検索関数を呼びます。

In [6]:
if call.name != "search_wikipedia_ja":
    raise ValueError("許可していないツール名です。")
query = call.args.get("query")
tool_output = search_wikipedia_ja(query)
print("検索語:", query)
print(tool_output)

検索語: 藤子・F・不二雄
{'status': 'ok', 'articles': [{'title': '藤子・F・不二雄', 'text': '藤子・F・不二雄（ふじこ エフ ふじお、1933年〈昭和8年〉12月1日 - 1996年〈平成8年〉9月23日）は、日本の漫画家。富山県高岡市定塚町出身。富山県立高岡工芸高等学校電気科卒業。本名は藤本 弘（ふじもと ひろし）。\n数多くの作品を発表し、児童漫画の新時代を築き、第一人者となる。独立を発表した1987年までは安孫子 素雄（独立後は藤子不二雄Ⓐ）と共に藤子不二雄として活動した。\n代表作は『オバケのQ太郎』（合作）、『ドラえもん』、『パーマン』（旧作は合作）、『キテレツ大百科』、『SF短編』シリーズである。「F」とは「藤本（フジモト）」の頭文字を意味する。死後、作品の著作権管理は生前に活動拠点としていた藤子・F・不二雄プロが担当している。', 'url': 'https://ja.wikipedia.org/wiki/%E8%97%A4%E5%AD%90%E3%83%BBF%E3%83%BB%E4%B8%8D%E4%BA%8C%E9%9B%84'}, {'title': '藤子不二雄', 'text': '藤子 不二雄（ふじこ ふじお）は、日本の漫画家である藤本弘（ふじもと ひろし、1933年〈昭和8年〉12月1日 - 1996年〈平成8年〉9月23日）と安孫子素雄（あびこ もとお、1934年〈昭和9年〉3月10日 - 2022年〈令和4年〉4月6日）が共同で1988年2月頃まで使っていたペンネーム。\n\n1951年に本名の連名「あびこもとお・ふじもとひろし」でプロデビュー。「足塚不二雄」を経て、1953年7月にペンネームを「藤子不二雄」とした。\n\n1987年12月に独立を発表し、1988年1月に独立パーティを開催。安孫子は「藤子不二雄Ⓐ」、藤本は1989年から「藤子・F・不二雄」のペンネームで活動を続けた（「#2つの闘病と独立」を参照）。\n\n活動初期の1954年の時点ですでに単独作品を多く執筆しており、合作を含む全作品を「藤子不二雄」という共同ペンネームで発表するという方式をとっていた（1988年の独立まで）。\n\n1965年に合作『オバケのQ太郎』がテレビアニメ化され大ブレイク。1966年には

### 検索結果をモデルへ返す

モデルの呼び出し要求に続けて、結果を `function_response` として返します。通信失敗や0件の場合も、その状態を渡します。

In [7]:
tool_answer = None
# 検索結果を、ツールの返り値 (function_response) として渡す
result_part = types.Part.from_function_response(
    name=call.name, response=tool_output
)
# 過去の入出力を含めて、会話履歴を構築
followup_contents = contents + [
    response.candidates[0].content,
    types.Content(role="user", parts=[result_part]),
]
# ツール出力を受けての続きを生成
final = client.models.generate_content(
    model=model_name, contents=followup_contents, config=config
)
tool_answer = final.text
print("検索後の回答:", tool_answer)

検索後の回答: 『ドラえもん』の作者である**藤子・F・不二雄**（本名：藤本 弘）の誕生日は、**1933年（昭和8年）12月1日**です。


## まとめ

今回見たツールとの組み合わせは LLM が自律的にツールの実行タイミング、実行方法を判断できることが特徴です。

LLM が自律的にツールを実行する方式以外にも、外部プログラムがツール実行タイミングなどを判断して、LLM と連携させることもできます。
